# Magic Formula tire models from Python

This notebook drives the Pacejka Magic Formula implementations (PAC2002 and MF 5.2) of the magic-formula C++ library through the `magic_formula` Python package. Inputs and outputs are NumPy arrays.

## Prerequisites

From the repository root:

```bash
conan install . --build=missing -s build_type=Release -s compiler.cppstd=26 -o with_benchmark=False
pip install "./python[notebook]"
```

## Units and conventions

- SI units throughout: N, m, m/s. Angles are in radians (use `np.deg2rad`).
- ISO-W sign convention; the rolling-resistance moment My is negative when rolling forward.
- Flag outputs (for example `sr_clamped`) are floats, 0.0 or 1.0.
- The kernel's working directory is the notebook folder, so the data files live at `../../tests/data`.

In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import magic_formula as mf

print(mf.__version__)
DATA = Path('../../tests/data')

## Loading and inspecting a tire

In [ ]:
tire = mf.load_tir(DATA / 'pac2002_S.tir')
print(repr(tire))
print('inputs :', tire.inputs)
print('outputs:', tire.outputs)
print('limits :', tire.limits)
print('fittyp :', tire.fittyp, ' use_mode:', tire.use_mode, ' tyre_side:', tire.tyre_side)

## Pure lateral slip

Inputs broadcast following NumPy rules: three loads (column) against 301 slip angles (row).

In [ ]:
alpha = np.deg2rad(np.linspace(-15, 15, 301))
fz = np.array([2000.0, 4000.0, 6000.0])[:, None]
res = tire.evaluate(fz=fz, sa=alpha, sr=0.0, ia=0.0, vx=16.0)

fig, ax = plt.subplots(figsize=(6, 4))
for i, load in enumerate(fz[:, 0]):
    ax.plot(np.rad2deg(alpha), res['fy'][i], label=f'Fz = {load:.0f} N')
ax.set_xlabel('Slip angle [deg]')
ax.set_ylabel('Lateral force Fy [N]')
ax.set_title('Pure lateral slip')
ax.legend()
ax.grid(True)
plt.show()

## Aligning moment

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for i, load in enumerate(fz[:, 0]):
    ax.plot(np.rad2deg(alpha), res['mz'][i], label=f'Fz = {load:.0f} N')
ax.set_xlabel('Slip angle [deg]')
ax.set_ylabel('Aligning moment Mz [N m]')
ax.set_title('Aligning moment, pure lateral slip')
ax.legend()
ax.grid(True)
plt.show()

## Pure longitudinal slip

In [ ]:
kappa = np.linspace(-0.3, 0.3, 301)
resx = tire.evaluate(fz=fz, sa=0.0, sr=kappa, ia=0.0, vx=16.0)

fig, ax = plt.subplots(figsize=(6, 4))
for i, load in enumerate(fz[:, 0]):
    ax.plot(kappa, resx['fx'][i], label=f'Fz = {load:.0f} N')
ax.set_xlabel('Slip ratio [-]')
ax.set_ylabel('Longitudinal force Fx [N]')
ax.set_title('Pure longitudinal slip')
ax.legend()
ax.grid(True)
plt.show()

## Combined slip

Fy against slip angle for several slip ratios, then the friction-circle plot over a slip-ratio by slip-angle grid.

In [ ]:
kappas = [0.0, 0.05, 0.1, 0.2]
fig, ax = plt.subplots(figsize=(6, 4))
for k in kappas:
    r = tire.evaluate(fz=4000.0, sa=alpha, sr=k, ia=0.0, vx=16.0)
    ax.plot(np.rad2deg(alpha), r['fy'], label=f'kappa = {k}')
ax.set_xlabel('Slip angle [deg]')
ax.set_ylabel('Lateral force Fy [N]')
ax.set_title('Combined slip, Fz = 4000 N')
ax.legend()
ax.grid(True)
plt.show()

kgrid = np.linspace(-0.3, 0.3, 61)
agrid = np.deg2rad(np.linspace(-12, 12, 61))
rc = tire.evaluate(fz=4000.0, sa=agrid[None, :], sr=kgrid[:, None], ia=0.0, vx=16.0)

fig, ax = plt.subplots(figsize=(5, 5))
for i in range(kgrid.size):
    ax.plot(rc['fy'][i, :], rc['fx'][i, :], color='C0', lw=0.5)
for j in range(agrid.size):
    ax.plot(rc['fy'][:, j], rc['fx'][:, j], color='C1', lw=0.5)
ax.set_xlabel('Lateral force Fy [N]')
ax.set_ylabel('Longitudinal force Fx [N]')
ax.set_title('Friction circle, Fz = 4000 N')
ax.set_aspect('equal')
ax.grid(True)
plt.show()

## Combined versus uncombined

With `combined=False` the combined-slip reduction is disabled, so the pure-slip force is returned regardless of the slip ratio.

In [ ]:
pure = mf.load_tir(DATA / 'pac2002_S.tir', combined=False)
r_comb = tire.evaluate(fz=4000.0, sa=alpha, sr=0.1, ia=0.0, vx=16.0)
r_pure = pure.evaluate(fz=4000.0, sa=alpha, sr=0.1, ia=0.0, vx=16.0)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(np.rad2deg(alpha), r_comb['fy'], label='combined')
ax.plot(np.rad2deg(alpha), r_pure['fy'], '--', label='uncombined')
ax.set_xlabel('Slip angle [deg]')
ax.set_ylabel('Lateral force Fy [N]')
ax.set_title('Fy at kappa = 0.1')
ax.legend()
ax.grid(True)
plt.show()

## Matrix API

`evaluate_batch` takes an `(N, n_inputs)` matrix with columns in `tire.inputs` order (FZ, SA, SR, IA, VX) and returns an `(N, n_outputs)` matrix. A Fortran-ordered float64 matrix avoids a copy.

In [ ]:
rng = np.random.default_rng(0)
n = 200_000
points = np.empty((n, 5), dtype=np.float64, order='F')
points[:, 0] = rng.uniform(1000.0, 7000.0, n)
points[:, 1] = rng.uniform(-0.2, 0.2, n)
points[:, 2] = rng.uniform(-0.2, 0.2, n)
points[:, 3] = rng.uniform(-0.05, 0.05, n)
points[:, 4] = rng.uniform(5.0, 40.0, n)

t0 = time.perf_counter()
out = tire.evaluate_batch(points)
dt = time.perf_counter() - t0
print('output shape:', out.shape)
print(f'{n / dt:,.0f} points/s ({dt * 1e3:.1f} ms)')

## MF 5.2

Loading an MF 5.2 file emits a `MagicFormulaWarning` (also stored on `TireModel.warnings`).

In [ ]:
mf52 = mf.load_tir(DATA / 'mf52_S52.tir')
print(mf52.name)
print(mf52.warnings)

r52 = mf52.evaluate(fz=4000.0, sa=alpha, sr=0.0, ia=0.0, vx=16.0)
r02 = tire.evaluate(fz=4000.0, sa=alpha, sr=0.0, ia=0.0, vx=16.0)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(np.rad2deg(alpha), r02['mz'], label='PAC2002')
ax.plot(np.rad2deg(alpha), r52['mz'], '--', label='MF 5.2')
ax.set_xlabel('Slip angle [deg]')
ax.set_ylabel('Aligning moment Mz [N m]')
ax.set_title('Mz, Fz = 4000 N')
ax.legend()
ax.grid(True)
plt.show()

## Clamp flags

Inputs beyond the validity range are clamped and flagged in the output rather than rejected. `pac2002_S_rng.tir` declares `KPUMAX = 1`, and `VXLOW` sets the low-speed threshold.

In [ ]:
rng_tire = mf.load_tir(DATA / 'pac2002_S_rng.tir')
print(rng_tire.limits)

sr = np.array([0.5, 1.0, 2.0, 10.0])
vx = np.array([0.5, 5.0, 10.0, 20.0])
rf = rng_tire.evaluate(fz=4000.0, sa=0.05, sr=sr, ia=0.0, vx=vx)
print('sr_clamped   :', rf['sr_clamped'])
print('below_vxlow  :', rf['below_vxlow'])

## Error handling

Invalid operating points raise `MagicFormulaError` carrying the error code, the offending key and the row index.

In [ ]:
try:
    tire.evaluate(fz=4000.0, sa=0.0, sr=0.0, ia=0.0, vx=np.array([10.0, -1.0]))
except mf.MagicFormulaError as e:
    print(e.code, e.key, e.row, e)

try:
    tire.evaluate(fz=4000.0, sa=0.0, sr=0.0, ia=0.0)
except TypeError as e:
    print('TypeError:', e)